In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as cp
from scipy.interpolate import griddata
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from scipy.interpolate import interp1d
from FallbackGen import FallbackGen
from TDECalculator import TDECalculator
import gc

In [2]:
MBH = 1e6
Rp = 6.6
a = -0.2
N = 5000
E = 1.0
Q = 0.0
orbit="rel"

In [3]:
mass_r = TDECalculator('MAMS2Msun', orbit, MBH, Rp, a, N=N)

: 

In [4]:
sample_r = mass_r.rel_whole_star_sample()

In [5]:
radii_r = sample_r['rr']

rtde = mass_r.R_TDE
Lz = mass_r.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_r <= 0.5,
    rtde - radii_r * mass_r.Rstar,
    rtde + radii_r * mass_r.Rstar
)

deltaE = mass_r.Rstar / mass_r.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_r['dEnergy_random'] * deltaE 
dLz = mass_r.dLz_random
dQ = mass_r.dQ_random
mass_ratio = mass_r.mass_ratio

In [6]:
dT_r = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, Q, dE, dLz, dQ, N)
np.save('dT_r.npy', dT_r.dTs)
gc.collect()

Computing radial periods for 116,280,000 particles ...
  E  range: [0.965684, 1.034317]
  Q  range: [-4.835e-04, 4.834e-04]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 11624/11624 (100%)
  Bound: 58,119,990 / 116,280,000
  Valid roots: 48,786,236
  Valid Lambda_r: 48,786,236
  Quadrature: 976 chunks ...
    chunk 976/976  (100%)
  Successful T_r: 48,786,236 / 116,280,000


31

In [7]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample['dEnergy_random']
    dT_rand = dT / delta
    dMass   = whole_star_sample['dMass']

    bins_E = np.linspace(-2, 2, 1000)
    bins_T = np.logspace(0, 6, 1000)

    # dE: use all particles with finite energy and mass
    valid_E = np.isfinite(dE_rand)
    hist_E, edges_E = np.histogram(dE_rand[valid_E], bins=bins_E,
                                   weights=dMass[valid_E], density=True)

    # dT: only particles with valid finite period within bin range
    valid_T = (np.isfinite(dT_rand) & np.isfinite(dE_rand)
               & (dT_rand >= 1.0) & (dT_rand <= 1e6))
    hist_T, edges_T = np.histogram(dT_rand[valid_T], bins=bins_T,
                                   weights=dMass[valid_T], density=True)

    return {"x": 0.5*(edges_E[:-1]+edges_E[1:]), "y": hist_E}, \
           {"x": 0.5*(edges_T[:-1]+edges_T[1:]), "y": hist_T}

In [8]:
DeltaE = mass_r.Rstar / mass_r.Rp**2
DeltaT = 1 / DeltaE**1.5

In [9]:
rel_r_E, rel_r_T = make_plot_dicts(sample_r, dT_r.dTs, DeltaT)

In [10]:
import json

adden = "m2_rp6p6_a0p2"

with open(f"Fallback_Data/rel_r_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_r_E.items()}, f)
with open(f"Fallback_Data/rel_r_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_r_T.items()}, f)